In [2]:
import pandas as pd
import datetime

orders = pd.read_csv("data/olist_orders_dataset.csv")
order_items = pd.read_csv("data/olist_order_items_dataset.csv")
customers = pd.read_csv("data/olist_customers_dataset.csv")
payments = pd.read_csv("data/olist_order_payments_dataset.csv")

orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

print("loaded")

loaded


In [3]:
# get delivered orders only with unique customer id
delivered = orders[orders["order_status"] == "delivered"]

delivered = delivered.merge(
    customers[["customer_id", "customer_unique_id"]], 
    on="customer_id"
)

# reference date - one day after the last order in dataset
reference_date = delivered["order_purchase_timestamp"].max() + datetime.timedelta(days=1)

print("reference date:", reference_date)
print("total delivered orders:", len(delivered))

reference date: 2018-08-30 15:00:37
total delivered orders: 96478


In [4]:
# calculate monetary value per order
order_value = payments.groupby("order_id")["payment_value"].sum().reset_index()
order_value.columns = ["order_id", "order_value"]

# merge with delivered orders
delivered = delivered.merge(order_value, on="order_id")

# calculate RFM
rfm = delivered.groupby("customer_unique_id").agg(
    recency=("order_purchase_timestamp", lambda x: (reference_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("order_value", "sum")
).reset_index()

rfm["monetary"] = rfm["monetary"].round(2)

print(rfm.head(10))
print("\ntotal customers:", len(rfm))

                 customer_unique_id  recency  frequency  monetary
0  0000366f3b9a7992bf8c76cfdf3221e2      112          1    141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f      115          1     27.19
2  0000f46a3911fa3c0805444483337064      537          1     86.22
3  0000f6ccb0745a6a4b88665a16c9f078      321          1     43.62
4  0004aac84e0df4da2b147fca70cf8255      288          1    196.89
5  0004bd2a26a76fe21f786e4fbd80607f      146          1    166.98
6  00050ab1314c0e55a6ca13cf7181fecf      132          1     35.38
7  00053a61a98854899e70ed204dd4bafe      183          1    419.18
8  0005e1862207bf6ccc02e4228effd9a0      543          1    150.12
9  0005ef4cd20d2893f0d9fbd94d3c0d97      170          1    129.76

total customers: 93357


In [5]:
# score each customer 1-5 for each dimension
rfm["R_score"] = pd.qcut(rfm["recency"], q=5, labels=[5,4,3,2,1])
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), q=5, labels=[1,2,3,4,5])
rfm["M_score"] = pd.qcut(rfm["monetary"], q=5, labels=[1,2,3,4,5])

rfm["R_score"] = rfm["R_score"].astype(int)
rfm["F_score"] = rfm["F_score"].astype(int)
rfm["M_score"] = rfm["M_score"].astype(int)

rfm["RFM_score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)

print(rfm[["customer_unique_id", "recency", "frequency", "monetary", "RFM_score"]].head(10))

                 customer_unique_id  recency  frequency  monetary RFM_score
0  0000366f3b9a7992bf8c76cfdf3221e2      112          1    141.90       414
1  0000b849f77a49e4a4ce2b2a4ca5be3f      115          1     27.19       411
2  0000f46a3911fa3c0805444483337064      537          1     86.22       112
3  0000f6ccb0745a6a4b88665a16c9f078      321          1     43.62       211
4  0004aac84e0df4da2b147fca70cf8255      288          1    196.89       214
5  0004bd2a26a76fe21f786e4fbd80607f      146          1    166.98       414
6  00050ab1314c0e55a6ca13cf7181fecf      132          1     35.38       411
7  00053a61a98854899e70ed204dd4bafe      183          1    419.18       315
8  0005e1862207bf6ccc02e4228effd9a0      543          1    150.12       114
9  0005ef4cd20d2893f0d9fbd94d3c0d97      170          1    129.76       413


In [7]:
def assign_segment(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]
    
    if r >= 4 and f >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "Recent Customers"
    elif r >= 3 and f <= 2:
        return "Potential Loyalists"
    elif r == 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2 and m >= 3:
        return "Cant Lose Them"
    elif r <= 2 and f <= 2:
        return "Lost"
    else:
        return "Others"

rfm["segment"] = rfm.apply(assign_segment, axis=1)

segment_summary = rfm.groupby("segment").agg(
    total_customers=("customer_unique_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean")
).reset_index()

segment_summary["avg_recency"] = segment_summary["avg_recency"].round(0)
segment_summary["avg_monetary"] = segment_summary["avg_monetary"].round(2)

print(segment_summary.sort_values("total_customers", ascending=False))

               segment  total_customers  avg_recency  avg_frequency  \
4      Loyal Customers            18824        169.0       1.034955   
7     Recent Customers            14984         91.0       1.000000   
2            Champions            14961         90.0       1.093242   
0              At Risk            11150        316.0       1.053632   
5               Others            11079        473.0       1.042332   
1       Cant Lose Them             8671        395.0       1.000000   
6  Potential Loyalists             7373        220.0       1.000000   
3                 Lost             6315        397.0       1.000000   

   avg_monetary  
4        161.33  
7        163.43  
2        176.96  
0        169.40  
5        164.54  
1        240.98  
6        154.01  
3         55.83  


In [8]:
segment_summary.to_csv("data/rfm_segments.csv", index=False)
rfm.to_csv("data/rfm_full.csv", index=False)

print("exported")

exported
